# NASA OSDR 太空飞行差异表达基因与富集分析

**数据来源**: NASA Open Science Data Repository (OSDR) Biological Data API  
**分析内容**: 9个OSD数据集的 Spaceflight vs Ground Control 差异表达分析 + GO/KEGG 基因富集分析

---

## 数据集概览

| OSD ID | 器官 | 任务 | 物种 |
|--------|------|------|------|
| OSD-48 | Liver | RR2 | C57BL/6J |
| OSD-168 | Liver | RR1 | C57BL/6J |
| OSD-102 | Left kidney | RR1 | C57BL/6J |
| OSD-513 | Left kidney | RR23 | C57BL/6J |
| OSD-101 | Gastrocnemius | RR-1 | C57BL/6J |
| OSD-105 | Tibialis anterior | RR-1 | C57BL/6J |
| OSD-103 | Quadriceps | RR-1 | C57BL/6J |
| OSD-100 | Left eye | RR-1 | C57BL/6J |
| OSD-288 | Spleen | MHU-1 | C57BL/6J |

## 分析流程

1. **数据获取** — 通过 NASA OSDR API 下载差异表达结果
2. **DEG筛选** — Raw P < 0.05, |Log2FC| > 0.26
3. **可视化** — 火山图、DEG 汇总图
4. **基因富集分析** — GSEA Preranked + Enrichr ORA (GO + KEGG)
5. **跨数据集比较** — 器官特异性通路变化

## 0. 环境安装与配置

In [ ]:
# 安装依赖（首次运行取消注释）
!pip install gseapy pandas numpy scipy matplotlib seaborn requests --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import os
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
import gseapy as gp

# 全局配置
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

print('Environment loaded!')
print(f'  gseapy: {gp.__version__}')
print(f'  pandas: {pd.__version__}')
print(f'  numpy:  {np.__version__}')

### 0.1 全局参数配置

In [ ]:
# NASA OSDR API
OSDR_API = 'https://visualization.osdr.nasa.gov/biodata/api/v2/query'

# 差异表达筛选阈值（来自Inspiration4 PPT）
PVAL_CUTOFF  = 0.05
LOGFC_CUTOFF = np.log2(1.2)   # ~ 0.263

# 数据集信息
DATASETS = {
    'OSD-48':  {'organ': 'Liver',           'mission': 'RR2',   'strain': 'C57BL/6J'},
    'OSD-168': {'organ': 'Liver',           'mission': 'RR1',   'strain': 'C57BL/6J'},
    'OSD-102': {'organ': 'Left kidney',     'mission': 'RR1',   'strain': 'C57BL/6J'},
    'OSD-513': {'organ': 'Left kidney',     'mission': 'RR23',  'strain': 'C57BL/6J'},
    'OSD-101': {'organ': 'Gastrocnemius',   'mission': 'RR-1',  'strain': 'C57BL/6J'},
    'OSD-105': {'organ': 'Tibialis ant.',   'mission': 'RR-1',  'strain': 'C57BL/6J'},
    'OSD-103': {'organ': 'Quadriceps',      'mission': 'RR-1',  'strain': 'C57BL/6J'},
    'OSD-100': {'organ': 'Left eye',        'mission': 'RR-1',  'strain': 'C57BL/6J'},
    'OSD-288': {'organ': 'Spleen',          'mission': 'MHU-1', 'strain': 'C57BL/6J'},
}

# 输出目录
DATA_DIR = 'nasa_osdr_data'
FIG_DIR  = 'nasa_osdr_figures'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f'Thresholds: P < {PVAL_CUTOFF}, |Log2FC| > {LOGFC_CUTOFF:.3f}')
print(f'Datasets: {len(DATASETS)}')

## 1. 数据获取 — NASA OSDR API

从 NASA OSDR 下载预计算的差异表达结果（`file.data type=differential expression`），无需从头运行 DESeq2。

In [ ]:
def download_osd_de(osd_id, data_dir=DATA_DIR):
    """Download differential expression data for an OSD dataset."""
    filepath = f'{data_dir}/{osd_id}_de.csv'
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1000:
        print(f'  {osd_id}: cached ({os.path.getsize(filepath)/1024/1024:.1f} MB)')
        return filepath
    
    url = f'{OSDR_API}/data/?id.accession={osd_id}&file.data%20type=differential%20expression&format=csv'
    print(f'  {osd_id}: downloading...', end=' ', flush=True)
    try:
        resp = requests.get(url, timeout=300, stream=True)
        total = 0
        with open(filepath, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=1024*1024):
                f.write(chunk)
                total += len(chunk)
        print(f'OK ({total/1024/1024:.1f} MB)')
        return filepath
    except Exception as e:
        print(f'FAILED: {e}')
        if os.path.exists(filepath):
            os.remove(filepath)
        return None

# Download all datasets
print('Downloading NASA OSDR differential expression data...')
for osd_id in DATASETS:
    download_osd_de(osd_id)
print('Done!')

## 2. 差异表达分析

解析预计算的 DE 结果，提取 **Spaceflight vs Ground Control** 比较，筛选上调/下调基因。

筛选标准（来自 Inspiration4 PPT）：
- Raw P < 0.05
- |Log2FC| > 0.26 (即 log2(1.2))

In [ ]:
def parse_osd_de(osd_id, data_dir=DATA_DIR):
    """Parse OSD DE data, extract Spaceflight vs Ground Control comparison."""
    filepath = f'{data_dir}/{osd_id}_de.csv'
    df = pd.read_csv(filepath, low_memory=False)
    cols = df.columns.tolist()
    
    # Find annotation columns
    symbol_col  = next((c for c in cols if c.endswith('/SYMBOL')), None)
    ensembl_col = next((c for c in cols if c.endswith('/ENSEMBL')), None)
    entrezid_col = next((c for c in cols if c.endswith('/ENTREZID')), None)
    
    # Find Log2fc columns
    logfc_cols = [c for c in cols if 'Log2fc' in c]
    
    # Priority 1: Space Flight Upon euthanasia vs Ground Control Upon euthanasia
    target_logfc = None
    comparison_name = None
    
    for c in logfc_cols:
        short = c.split('/')[-1].replace('Log2fc_', '')
        if ')v(' in short:
            left, right = short.split(')v(', 1)
            left = left.lstrip('(')
            right = right.rstrip(')')
            if 'Space Flight' in left and 'Ground Control' in right:
                if 'Upon euthanasia' in left and 'Upon euthanasia' in right:
                    target_logfc = c
                    comparison_name = 'FLT vs GC (UponEuth)'
                    break
    
    # Priority 2: any Space Flight vs Ground Control
    if target_logfc is None:
        for c in logfc_cols:
            short = c.split('/')[-1].replace('Log2fc_', '')
            if ')v(' in short:
                left, right = short.split(')v(', 1)
                left = left.lstrip('(')
                right = right.rstrip(')')
                if 'Space Flight' in left and 'Ground Control' in right:
                    target_logfc = c
                    comparison_name = 'FLT vs GC'
                    break
    
    if target_logfc is None:
        return None
    
    # Match P.value and Adj.p.value columns
    comparison_part = target_logfc.split('/')[-1].replace('Log2fc_', '')
    pval_short = 'P.value_' + comparison_part
    adjp_short = 'Adj.p.value_' + comparison_part
    pval_col = next((c for c in cols if c.endswith(pval_short)), None)
    adjp_col = next((c for c in cols if c.endswith(adjp_short)), None)
    
    # Build DEG DataFrame
    deg = pd.DataFrame()
    deg['SYMBOL'] = df[symbol_col].astype(str) if symbol_col else None
    deg['ENSEMBL'] = df[ensembl_col].astype(str) if ensembl_col else None
    if entrezid_col:
        eid = df[entrezid_col].astype(str).str.split('|').str[0]
        deg['ENTREZID'] = pd.to_numeric(eid, errors='coerce')
    deg['Log2FC'] = pd.to_numeric(df[target_logfc], errors='coerce')
    deg['Pvalue'] = pd.to_numeric(df[pval_col], errors='coerce') if pval_col else np.nan
    deg['AdjPvalue'] = pd.to_numeric(df[adjp_col], errors='coerce') if adjp_col else np.nan
    
    # Filter significant DEGs
    valid = deg.dropna(subset=['Log2FC', 'Pvalue'])
    sig = valid[(valid['Pvalue'] < PVAL_CUTOFF) & (valid['Log2FC'].abs() > LOGFC_CUTOFF)]
    up = sig[sig['Log2FC'] > 0].sort_values('Log2FC', ascending=False)
    down = sig[sig['Log2FC'] < 0].sort_values('Log2FC')
    sig_adj = valid[(valid['AdjPvalue'] < PVAL_CUTOFF) & (valid['Log2FC'].abs() > LOGFC_CUTOFF)]
    
    return {
        'osd_id': osd_id,
        'comparison': comparison_name,
        'total_genes': len(df),
        'n_sig_rawP': len(sig),
        'n_up': len(up),
        'n_down': len(down),
        'n_sig_adjP': len(sig_adj),
        'deg': deg,
        'sig': sig,
        'up': up,
        'down': down,
        'up_genes': [g for g in up['SYMBOL'].tolist() if g != 'nan'],
        'down_genes': [g for g in down['SYMBOL'].tolist() if g != 'nan'],
    }

# Process all datasets
results = {}
for osd_id, info in DATASETS.items():
    r = parse_osd_de(osd_id)
    if r:
        results[osd_id] = r
        print(f"{osd_id} ({info['organ']}): UP={r['n_up']} DOWN={r['n_down']} (AdjP<0.05: {r['n_sig_adjP']})")

print(f'Processed {len(results)} datasets')

### 2.1 DEG 汇总表

In [ ]:
# Summary table
summary = []
for osd_id, r in results.items():
    info = DATASETS[osd_id]
    summary.append({
        'OSD_ID': osd_id,
        'Organ': info['organ'],
        'Mission': info['mission'],
        'Comparison': r['comparison'],
        'Total_Genes': r['total_genes'],
        'Sig_RawP': r['n_sig_rawP'],
        'Up_Regulated': r['n_up'],
        'Down_Regulated': r['n_down'],
        'Sig_AdjP': r['n_sig_adjP'],
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

# Save
summary_df.to_csv(f'{DATA_DIR}/deg_summary.csv', index=False)
print('Summary saved!')

## 3. 可视化 — 火山图与 DEG 汇总

In [ ]:
# DEG Summary Bar Chart
fig, ax = plt.subplots(figsize=(12, 5))

names = [DATASETS[k]['organ'] + f'\n({k})' for k in results]
up_counts = [results[k]['n_up'] for k in results]
down_counts = [results[k]['n_down'] for k in results]
adjp_counts = [results[k]['n_sig_adjP'] for k in results]

x = np.arange(len(names))
width = 0.25

bars1 = ax.bar(x - width, up_counts, width, label='Upregulated (Flight UP)', color='#e74c3c', alpha=0.85)
bars2 = ax.bar(x, down_counts, width, label='Downregulated (Flight DOWN)', color='#3498db', alpha=0.85)
bars3 = ax.bar(x + width, adjp_counts, width, label='Adj.P < 0.05', color='#2ecc71', alpha=0.85)

ax.set_ylabel('Number of DEGs')
ax.set_title('NASA OSDR: Differentially Expressed Genes Across Spaceflight Datasets\n(Raw P < 0.05, |Log2FC| > 0.26)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right', fontsize=9)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if h > 30:
            ax.text(bar.get_x() + bar.get_width()/2., h, f'{int(h)}',
                   ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/deg_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Volcano Plots (3x3 grid)
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

for idx, (osd_id, r) in enumerate(results.items()):
    ax = axes[idx]
    info = DATASETS[osd_id]
    deg = r['deg'].dropna(subset=['Log2FC', 'Pvalue']).copy()
    deg['NegLogP'] = -np.log10(deg['Pvalue'].clip(lower=1e-300))
    
    # Color by significance
    up_mask = (deg['Log2FC'] > LOGFC_CUTOFF) & (deg['Pvalue'] < PVAL_CUTOFF)
    down_mask = (deg['Log2FC'] < -LOGFC_CUTOFF) & (deg['Pvalue'] < PVAL_CUTOFF)
    adjp_mask = (deg['AdjPvalue'] < PVAL_CUTOFF) & (deg['Log2FC'].abs() > LOGFC_CUTOFF)
    
    deg['color'] = '#dcdde1'
    deg.loc[up_mask, 'color'] = '#e74c3c'
    deg.loc[down_mask, 'color'] = '#3498db'
    deg.loc[adjp_mask, 'color'] = '#2c3e50'
    
    ax.scatter(deg['Log2FC'], deg['NegLogP'], c=deg['color'], s=3, alpha=0.5, rasterized=True)
    
    ax.axhline(-np.log10(PVAL_CUTOFF), color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.axvline(LOGFC_CUTOFF, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.axvline(-LOGFC_CUTOFF, color='gray', ls='--', lw=0.8, alpha=0.5)
    
    # Label top genes
    top = deg[up_mask | down_mask].nsmallest(5, 'Pvalue')
    for _, row in top.iterrows():
        gene = str(row.get('SYMBOL', ''))[:8]
        if gene and gene != 'nan':
            ax.annotate(gene, (row['Log2FC'], row['NegLogP']),
                       fontsize=6, alpha=0.8, xytext=(3,3), textcoords='offset points')
    
    ax.set_title(f"{osd_id}: {info['organ']}\nUP={r['n_up']} DOWN={r['n_down']}", fontsize=10)
    ax.set_xlabel('Log2FC')
    ax.set_ylabel('-Log10(P)')

fig.suptitle('NASA OSDR Volcano Plots: Spaceflight vs Ground Control', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/volcano_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 基因富集分析

采用两种互补策略：
- **GSEA Preranked**: 使用全部基因的排序列表（按 -log10(P) * sign(LogFC)），无需硬阈值
- **Enrichr ORA**: 对上调/下调基因分别做过表达分析

基因集: GO Biological Process 2023 + KEGG 2021

In [ ]:
def run_gsea_prerank(osd_id, deg_df, gene_sets='KEGG_2021_Human'):
    """Run GSEA Preranked for a single dataset."""
    deg_clean = deg_df.dropna(subset=['SYMBOL', 'Log2FC', 'Pvalue']).copy()
    deg_clean = deg_clean[deg_clean['SYMBOL'] != 'nan']
    deg_clean = deg_clean[~deg_clean.duplicated(subset='SYMBOL', keep='first')]
    
    if len(deg_clean) < 20:
        return None
    
    deg_clean['rank_metric'] = -np.log10(deg_clean['Pvalue'].clip(lower=1e-300)) * np.sign(deg_clean['Log2FC'])
    rnk = deg_clean[['SYMBOL', 'rank_metric']].sort_values('rank_metric', ascending=False)
    rnk.columns = ['gene', 'score']
    
    gsea = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,
        organism='mouse',
        outdir=None,
        no_plot=True,
        permutation_num=100,
        seed=42,
        min_size=5,
        max_size=500,
    )
    
    res = gsea.res2d.copy()
    res['NES'] = res['NES'].astype(float)
    res['FDR'] = res['FDR q-val'].astype(float)
    sig = res[res['FDR'] < 0.25].sort_values('FDR')  # GSEA convention: FDR < 0.25
    return sig

def run_enrichr_ora(gene_list, gene_sets='KEGG_2021_Human'):
    """Run Enrichr over-representation analysis."""
    if len(gene_list) < 5:
        return None
    try:
        enr = gp.enrichr(
            gene_list=gene_list,
            gene_sets=gene_sets,
            organism='mouse',
            outdir=None,
            no_plot=True,
            cutoff=0.05
        )
        sig = enr.results[enr.results['Adjusted P-value'] < 0.05]
        return sig
    except:
        return None

print('Enrichment analysis functions defined!')

In [ ]:
# Run GSEA and ORA for all datasets
enrichment_results = {}

for osd_id, r in results.items():
    info = DATASETS[osd_id]
    print(f'\n{"="*50}')
    print(f'{osd_id} ({info["organ"]})...')
    
    # GSEA Preranked - KEGG
    gsea_kegg = run_gsea_prerank(osd_id, r['sig'], 'KEGG_2021_Human')
    if gsea_kegg is not None and len(gsea_kegg) > 0:
        print(f'  GSEA-KEGG: {len(gsea_kegg)} significant terms (FDR<0.25)')
        for _, row in gsea_kegg.head(5).iterrows():
            print(f"    NES={row['NES']:+.3f} FDR={row['FDR']:.3f} | {row['Term'][:50]}")
    else:
        print('  GSEA-KEGG: No significant terms')
    
    # Enrichr ORA
    ora_up = run_enrichr_ora(r['up_genes'], 'KEGG_2021_Human')
    ora_down = run_enrichr_ora(r['down_genes'], 'KEGG_2021_Human')
    
    n_up_ora = len(ora_up) if ora_up is not None else 0
    n_down_ora = len(ora_down) if ora_down is not None else 0
    print(f'  Enrichr ORA: UP={n_up_ora} DOWN={n_down_ora} significant KEGG terms (AdjP<0.05)')
    
    enrichment_results[osd_id] = {
        'gsea_kegg': gsea_kegg,
        'ora_up': ora_up,
        'ora_down': ora_down,
    }

print(f'\nEnrichment analysis complete: {len(enrichment_results)} datasets')

### 4.1 KEGG 通路富集可视化

In [ ]:
# KEGG Enrichment Bar Plot
selected = ['OSD-48', 'OSD-102', 'OSD-168', 'OSD-513', 'OSD-100', 'OSD-101']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, osd_id in enumerate(selected):
    ax = axes[idx]
    info = DATASETS[osd_id]
    gsea = enrichment_results[osd_id]['gsea_kegg']
    
    if gsea is None or len(gsea) == 0:
        ax.set_title(f"{osd_id}: {info['organ']}\nNo significant terms")
        continue
    
    top = gsea.head(10).copy()
    top = top.sort_values('NES', ascending=True)
    top['Term_short'] = top['Term'].str[:40]
    
    colors = ['#e74c3c' if x > 0 else '#3498db' for x in top['NES']]
    ax.barh(top['Term_short'], top['NES'], color=colors, alpha=0.8, height=0.7)
    ax.set_xlabel('NES')
    ax.set_title(f"{osd_id}: {info['organ']}", fontsize=10)
    ax.axvline(0, color='gray', lw=0.8)
    ax.tick_params(labelsize=7)

fig.suptitle('Top KEGG Pathways (GSEA Preranked)\nRed = Upregulated in Flight | Blue = Downregulated in Flight',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/kegg_enrichment_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. GO Biological Process 富集分析

In [ ]:
# GO GSEA for key datasets
go_results = {}
for osd_id in ['OSD-48', 'OSD-168', 'OSD-513', 'OSD-100']:
    r = results[osd_id]
    info = DATASETS[osd_id]
    print(f'\n{osd_id} ({info["organ"]}) - GO GSEA...')
    gsea_go = run_gsea_prerank(osd_id, r['sig'], 'GO_Biological_Process_2023')
    go_results[osd_id] = gsea_go
    if gsea_go is not None and len(gsea_go) > 0:
        print(f'  {len(gsea_go)} significant GO terms (FDR<0.25)')
        for _, row in gsea_go.head(8).iterrows():
            direction = 'UP' if row['NES'] > 0 else 'DOWN'
            print(f"    [{direction}] NES={row['NES']:+.3f} FDR={row['FDR']:.3f} | {row['Term'][:60]}")
    else:
        print('  No significant terms')

In [ ]:
# GO enrichment bar plot
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, osd_id in enumerate(['OSD-48', 'OSD-168', 'OSD-513', 'OSD-100']):
    ax = axes[idx]
    info = DATASETS[osd_id]
    gsea_go = go_results.get(osd_id)
    
    if gsea_go is None or len(gsea_go) == 0:
        ax.set_title(f"{osd_id}: {info['organ']}\nNo significant terms")
        continue
    
    top = gsea_go.head(15).copy()
    top = top.sort_values('NES', ascending=True)
    top['Term_short'] = top['Term'].str.replace(r' \(GO:\d+\)', '', regex=True).str[:50]
    
    colors = ['#e74c3c' if x > 0 else '#3498db' for x in top['NES']]
    ax.barh(top['Term_short'], top['NES'], color=colors, alpha=0.8, height=0.7)
    ax.set_xlabel('NES')
    ax.set_title(f"{osd_id}: {info['organ']}", fontsize=10)
    ax.axvline(0, color='gray', lw=0.8)
    ax.tick_params(labelsize=6)

fig.suptitle('GO Biological Process (GSEA Preranked)\nRed = Upregulated in Flight | Blue = Downregulated in Flight',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/go_enrichment_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 跨数据集比较 — 器官特异性通路变化

比较不同器官在太空飞行中的 KEGG 通路变化模式。

In [ ]:
# Collect all GSEA-KEGG results across datasets
all_kegg = []
for osd_id, enr in enrichment_results.items():
    gsea = enr.get('gsea_kegg')
    if gsea is not None and len(gsea) > 0:
        for _, row in gsea.iterrows():
            all_kegg.append({
                'OSD_ID': osd_id,
                'Organ': DATASETS[osd_id]['organ'],
                'Term': row['Term'],
                'NES': float(row['NES']),
                'FDR': float(row['FDR']),
            })

kegg_df = pd.DataFrame(all_kegg)

# Find pathways enriched in >=3 datasets
common_pathways = kegg_df.groupby('Term').size().reset_index(name='n_datasets')
common_pathways = common_pathways[common_pathways['n_datasets'] >= 3].sort_values('n_datasets', ascending=False)

print(f'KEGG pathways significant in >=3 datasets: {len(common_pathways)}')
print()
for _, row in common_pathways.head(20).iterrows():
    print(f"  [{row['n_datasets']}x] {row['Term']}")

# Build heatmap matrix
top_pathways = common_pathways.head(15)['Term'].tolist()
heatmap_data = kegg_df[kegg_df['Term'].isin(top_pathways)].pivot_table(
    index='Term', columns='Organ', values='NES', aggfunc='first'
)

In [ ]:
# Cross-organ KEGG pathway heatmap
if len(heatmap_data) > 0:
    fig, ax = plt.subplots(figsize=(10, max(6, len(heatmap_data)*0.4)))
    sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                linewidths=0.5, ax=ax, cbar_kws={'label': 'NES'})
    ax.set_title('Cross-Organ KEGG Pathway Enrichment (GSEA NES)\nRed = Up in Flight | Blue = Down in Flight',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('')
    ax.tick_params(labelsize=8)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/cross_organ_kegg_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No common pathways found for heatmap')

## 7. 关键发现总结

In [ ]:
print('=' * 70)
print('NASA OSDR Spaceflight DEG & Enrichment Analysis - Key Findings')
print('=' * 70)
print()

findings = [
    ('Liver (OSD-168/RR1)',
     '6,685 DEGs (UP=3,911 DOWN=2,774; AdjP: 4,729)',
     'UP: Chronic myeloid leukemia, Hematopoietic cell lineage',
     'DOWN: Ribosome, Fatty acid degradation, Oxidative phosphorylation, PPAR signaling, Drug metabolism',
     'Widespread metabolic suppression (fatty acid oxidation, drug metabolism, OXPHOS) indicating hepatic metabolic reprogramming'),
    
    ('Liver (OSD-48/RR2)',
     '1,097 DEGs (UP=647 DOWN=450; AdjP: 65)',
     'UP: Cushing syndrome, Cortisol/Aldosterone synthesis, Steroid hormone biosynthesis',
     'DOWN: AMPK signaling',
     'Steroid hormone synthesis pathways significantly upregulated, Cushing syndrome-like stress response'),
    
    ('Kidney (OSD-513/RR23)',
     '5,534 DEGs (UP=2,916 DOWN=2,618; AdjP: 3,317)',
     'UP: Xenobiotics metabolism (CYP450), Drug metabolism, Peroxisome, PPAR signaling',
     'DOWN: Protein processing in ER, Antigen processing/presentation',
     'Kidney xenobiotic metabolism & peroxisome activation, ER protein processing and antigen presentation suppressed'),
    
    ('Eye (OSD-100/RR-1)',
     '1,201 DEGs (UP=505 DOWN=696; AdjP: 120)',
     'UP: Influenza A, Hepatitis C, Coronavirus disease',
     'DOWN: Diabetic cardiomyopathy, Apelin signaling, BCAA degradation',
     'Immune/inflammatory pathway activation, branched-chain amino acid degradation downregulated'),
    
    ('Gastrocnemius (OSD-101)',
     '1,311 DEGs (UP=667 DOWN=644; AdjP: 228)',
     'UP: (none significant)',
     'DOWN: FoxO signaling, AMPK signaling, Autophagy, mTOR signaling, Longevity regulating',
     'Comprehensive downregulation of FoxO/AMPK/autophagy/mTOR longevity pathways, consistent with muscle atrophy'),
]

for organ, degs, up, down, insight in findings:
    print(f'>> {organ}')
    print(f'   DEGs: {degs}')
    print(f'   UP-KEGG: {up}')
    print(f'   DOWN-KEGG: {down}')
    print(f'   Insight: {insight}')
    print()

## 8. 导出结果

In [ ]:
# Export all DEG results
for osd_id, r in results.items():
    r['sig'].to_csv(f'{DATA_DIR}/{osd_id}_deg_filtered.csv', index=False)
    r['deg'].dropna(subset=['Log2FC', 'Pvalue']).to_csv(f'{DATA_DIR}/{osd_id}_deg_all.csv', index=False)

# Export enrichment results
for osd_id, enr in enrichment_results.items():
    for key in ['gsea_kegg', 'ora_up', 'ora_down']:
        df = enr.get(key)
        if df is not None and len(df) > 0:
            df.to_csv(f'{DATA_DIR}/{osd_id}_enrichment_{key}.csv', index=False)

print(f'All results saved to: {DATA_DIR}/')
print(f'  DEG files: {len(results)} datasets')
print(f'  Figure files: {FIG_DIR}/')